<a href="https://colab.research.google.com/github/Swastika200105/My-daily-data-science-journal/blob/main/Data_analysis_of_student_attendance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart,LineChart, PieChart, Reference
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

df = pd.read_excel("/content/sample_data/Student_Attendance_Big_Dataset.xlsx")
df["Date"] = pd.to_datetime(df["Date"])# convert Date column  to actual datetime
#create present_flag
df["Present_Flag"] = (df["Status"].str.strip().str.lower() == "present").astype(int)
# create absent_flag
df["Absent_Flag"] = (df["Status"].str.strip().str.lower() == "absent").astype(int)


# Core KPI calculations
total_records = len(df)#Total number of attendance records
students =  df["Student_ID"].nunique() #Number of unique students
dates = df["Date"].nunique() #Number of unique dates
present = df["Present_Flag"].sum() #Total number of present records
absent = df["Absent_Flag"].sum() #Total number of absent records
attendance_rate = (present / total_records) * 100 #attendance rate calculation

# Summary tables

class_summary = df.groupby("Class").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
class_summary["Attendance_Rate_%"] = class_summary["Present"] / class_summary["total_records"] * 100
class_summary = class_summary.sort_values(by="Attendance_Rate_%", ascending=False)

section_summary = df.groupby("Section").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
section_summary["Attendance_Rate_%"] = section_summary["Present"] / section_summary["total_records"] * 100
section_summary = section_summary.sort_values(by="Attendance_Rate_%", ascending=False)

monthly = df.assign(Month=df["Date"].dt.to_period("M")).groupby("Month").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
monthly["Attendance_Rate_%"] = monthly["Present"] / monthly["total_records"] * 100
daily = df.groupby("Date").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
daily["Attendance_Rate_%"] = daily["Present"] / daily["total_records"] * 100
daily = daily.sort_values("Date")

student_summary = df.groupby(["Student_ID", "Student_Name"]).agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
student_summary["Attendance_Rate_%"] = student_summary["Present"] / student_summary["total_records"] * 100
student_summary["Risk_Level"] = np.select (
  [
       student_summary["Attendance_Rate_%"] < 75,
       student_summary["Attendance_Rate_%"] < 85,
       student_summary["Attendance_Rate_%"] < 90
  ],
  ["Critical", "High", "Watch"],
  default = "Good"
)

student_summary = student_summary.sort_values(["Attendance_Rate_%", "Absent"], ascending=[True, False])


class_section = df.groupby(["Class", "Section"]).agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
class_section["Attendance_Rate_%"] = class_section["Present"] / class_section["total_records"] * 100
class_section = class_section.sort_values("Attendance_Rate_%", ascending= False)


weekday = df.assign(Weekday=df["Date"].dt.day_name()).groupby("Weekday").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
weekday["Attendance_Rate_%"] = weekday["Present"] / weekday["total_records"] * 100
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday["Weekday"] = pd.Categorical(weekday["Weekday"], categories=weekday_order, ordered=True)
weekday = weekday.sort_values("Weekday")
weekday["Weekday"] = weekday["Weekday"].astype(str)

# Data quality assessment
missing = df.isna().sum().reset_index()
missing.columns = ["Column", " Missing Values"]
duplicate_rows = int(df.duplicated(subset=["Student_ID", "Date"]).sum())
student_name_conflicts = int(df.groupby("Student_ID")["Student_Name"].nunique().gt(1).sum())
student_class_changes = int(df.groupby("Student_ID")["Class"].nunique().gt(1).sum())
student_section_changes = int(df.groupby("Student_ID")["Section"].nunique().gt(1).sum())



In [50]:
quality = pd.DataFrame({
    "Check": [
        "total records", "Unique students", "Unique dates",
        "Duplicate Student_ID + Date records", "Student IDs with multiple names",
        "Student IDs appearing in multiple classes",
        "Student IDs appearing in multiple sections",
        "Missing values (all columns combined)",
        "Distinct Status values"
    ],
    "Result": [
        total_records, students, dates, duplicate_rows, student_name_conflicts,
        student_class_changes, student_section_changes, int(missing[" Missing Values"].sum()),
        ", ".join(sorted(df["Status"].dropna().astype(str).unique()))
    ],
    "Assessment": [
        "OK", "OK", "OK",
        "OK" if duplicate_rows == 0 else "Review",
        "OK" if student_name_conflicts == 0 else "Review",
        "Review — likely synthetic/inconsistent class assignment",
        "Review — likely synthetic/inconsistent section assignment",
        "OK" if missing[" Missing Values"].sum() == 0 else "Review",
        "OK"
    ]
})

In [51]:
# Workbook creation
with pd.ExcelWriter("/content/sample_data/Professional_Attendance_Big_Dataset.xlsx", engine="openpyxl") as writer:
  # Executive sheets
  kpi = pd.DataFrame({
      "KPI": [
          "Total Attendance Records", "Unique Students", "Attendance Days",
          "Present Records", "Absent Records", "Overall Attendance Rate (%)",
          "Data Range"
      ],
      "Value": [
          total_records, students, dates,
          present, absent, round(attendance_rate, 2),
          f"{df['Date'].min()} - {df['Date'].max()}"

      ]
  })
  kpi.to_excel(writer, sheet_name="Executive Summary", index=False, startrow=2)
  class_summary.to_excel(writer, sheet_name="Class Analysis", index=False)
  section_summary.to_excel(writer, sheet_name="Section Analysis", index=False)
  monthly.to_excel(writer, sheet_name="Monthly Trend", index=False)
  daily.to_excel(writer, sheet_name="Daily Trend", index=False)
  student_summary.to_excel(writer, sheet_name="Student Risk", index=False)
  class_section.to_excel(writer, sheet_name="Class-Section", index=False)
  weekday.to_excel(writer, sheet_name="Weekday Analysis", index=False)
  quality.to_excel(writer, sheet_name="Data Quality", index=False)



In [52]:
# keep cleaned dataset separately, without helper flags
cleaned = df[["Student_ID", "Student_Name", "Class", "Section", "Date", "Status"]].copy()
cleaned.to_excel(writer, sheet_name="Cleaned Data", index=False)


# Formatting + charts
wb = load_workbook("/content/sample_data/Professional_Attendance_Big_Dataset.xlsx")

# Palette
navy = "1F4E78"
blue = "5B9BD5"
light_blue = "D9EAF7"
green = "70AD47"
orange = "ED7D31"
red = "C00000"
gray = "E7E6E6"
white = "FFFFFF"

thin = Side(style="thin", color="D9E1F2")

for ws in wb.worksheets:
  ws.freeze_panes = "A2"
  ws.auto_filter.ref = ws.dimensions

  for cell in ws[1]:
    cell.font = Font(bold=True, color=white)
    cell.fill = PatternFill("solid", fgColor=navy)
    cell.alignment = Alignment(horizontal="center", vertical="center")

  for row in ws.iter_rows():
    for cell in row:
      cell.border = Border(bottom=thin)
      cell.alignment = Alignment(vertical="center")
  #widths
  for col_cells in ws.columns:
    max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
    ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(max_len +2, 12), 28)


In [53]:
# Executive Summary special formatting
ws = wb["Executive Summary"]
ws["A1"] = "Professional Student Attendance Analysis"
ws["A1"].font = Font(bold=True, size=18, color=white)
ws["A1"].fill = PatternFill("solid", fgColor=navy)
ws.merge_cells("A1:B1")
ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
for c in ws[3]:
  c.font = Font(bold=True, color=white)
  c.fill = PatternFill("solid", fgColor=navy)
for r in range (4, 11):
  ws[f"A{r}"].font = Font(bold=True)
  ws.column_dimensions["A"].width = 32
  ws.column_dimensions["B"].width = 32

# Number formatting for analytical sheets
percent_sheets = ["Class Analysis", "Section Analysis", "Monthly Trend",
                  "Daily Trend", "Student Risk", "Class-Section", "Weekday Analysis"]
for s in percent_sheets:
  ws = wb[s]
  for row in ws.iter_rows(min_row=2):
    for cell in row:
      if isinstance(cell.value, (int, float)) and cell.column == ws.max_column:
        cell.number_format = "0.00"





In [54]:
# conditional formatting-like manual highlighting for student risk

ws = wb["Student Risk"]
headers = {c.value: c.column for c in ws[1]}
risk_col = headers["Risk_Level"]
rate_col = headers["Attendance_Rate_%"]
for r in range (2, ws.max_row + 1):
  risk = ws.cell(r, risk_col).value
  if risk == "Critical":
    ws.cell(r, risk_col).fill = PatternFill("solid", fgColor="F4CCCC")
    ws.cell(r, rate_col).fill = PatternFill("solid", fgColor="F4CCCC")
  elif risk == "High":
    ws.cell(r, risk_col).fill = PatternFill("solid", fgColor="FCE5CD")
    ws.cell(r, rate_col).fill = PatternFill("solid", fgColor="FCE5CD")
  elif risk == "Watch":
    ws.cell(r, risk_col).fill = PatternFill("solid", fgColor="FFF2CC")

# Add charts
ws = wb["Executive Summary"]

# Class chart
chart = BarChart()
chart.title = "Attendance Rate by Class"
chart.y_axis.title = "Attendance Rate (%)"
chart.x_axis.title = "Class"
data = Reference(wb["Class Analysis"], min_col=5, min_row=1, max_row=6)
cats = Reference(wb["Class Analysis"], min_col=1, min_row=2, max_row=6)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.height = 7
chart.width = 12
ws.add_chart(chart, "D3")


# Monthly trend chart
line = LineChart()
line.title = "Monthly Attendance Trend"
line.y_axis.title = "Attendance Rate (%)"
line.x_axis.title = "Month"
data = Reference(wb["Monthly Trend"], min_col=5, min_row=1, max_row=5)
cats = Reference(wb["Monthly Trend"], min_col=1, min_row=2, max_row=5)
line.add_data(data, titles_from_data=True)
line.set_categories(cats)
line.height = 7
line.width = 12
ws.add_chart(line, "D18")

# Status pie
pie = PieChart()
pie.title = "Present vs Absent"
data = Reference(wb["Executive Summary"], min_col=2, min_row=7, max_row=8)
labels = Reference(wb["Executive Summary"], min_col=1, min_row=7, max_row=8)
pie.add_data(data, titles_from_data=False)
pie.set_categories(labels)
pie.height = 7
pie.width = 10
ws.add_chart(pie, "Q3")


In [60]:
# Data Quality title/notes
ws = wb["Data Quality"]
ws["E1"] = "Data Quality Assessment"
ws["E1"].font = Font(bold=True, size=18, color=white)
ws["E1"].fill = PatternFill("solid", fgColor=navy)
notes = [
    "No missing values were detected.",
    "No duplicate Student_Id + Data attendance records were detected.",
    "Each of the 200 students appears accross all 5 classes and all 3 sections.",
    "This class/section instability is a major data-quality concern if students are expected to have fixed class assignments.",
    "The dataset therefore appears synthetic or intentionally randomized rather than a real school roaster."

]
for i, note in enumerate(notes, start=2):
  ws[f"E{i}"] = note
  ws[f"E{i}"].alignment = Alignment(wrap_text=True, vertical="top")
  ws.column_dimensions["E"].width = 70

 # Add a methodology sheet
methodology = wb.create_sheet("Methodology")
methodology_rows = [
     ["Analysis Area", "Method"],
     ["Data validation", "Checked shape, data types, missing values, duplicates, unique entities, and categorical values."],
    ["Attendance KPI", "Attendance Rate = Present Records / Total Attendance Records × 100."],
    ["Class analysis", "Aggregated attendance by Grade/Class and compared present, absent, and attendance rate."],
    ["Section analysis", "Aggregated attendance by Section."],
    ["Trend analysis", "Calculated monthly and daily attendance rates over the available date range."],
    ["Student risk", "Calculated student-level attendance rate and classified <75% as Critical, 75–<85% High, 85–<90% Watch, ≥90% Good."],
    ["Data quality", "Checked whether Student_ID consistently maps to Student_Name, Class, and Section."],
    ["Business interpretation", "Highlighted attendance performance and structural issues that affect real-world reliability."]
]

for row in methodology_rows:
    methodology.append(row)
for c in methodology[1]:
    c.font = Font(bold=True, color=white)
    c.fill = PatternFill("solid", fgColor=navy)
methodology.column_dimensions["A"].width = 24
methodology.column_dimensions["B"].width = 100
for row in methodology.iter_rows():
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

# Save
wb.save("/content/sample_data/Professional_Attendance_Big_Dataset.xlsx")

print(f"Created: {"/content/sample_data/Professional_Attendance_Big_Dataset.xlsx"}")
print(f"Overall attendance rate: {attendance_rate:.2f}%")
print(f"Present: {present:,} | Absent: {absent:,}")
print("Key data-quality issue: every student appears across all classes and sections.")

Created: /content/sample_data/Professional_Attendance_Big_Dataset.xlsx
Overall attendance rate: 90.16%
Present: 21,638 | Absent: 2,362
Key data-quality issue: every student appears across all classes and sections.
